# Mixed-data classification pipeline

Select the `.venv-py311` kernel.

Predict whether a customer will churn in the next 90 days using information available at the review date. Each row represents one customer. `churn=1` means the customer leaves; `churn=0` means they stay.

The 500 customers below are synthetic, generated for practising missing values, mixed feature types, ColumnTransformer and model evaluation. Performance on this dataset describes this simulation.

We will inspect, split, build separate numeric and categorical pipelines, combine them with ColumnTransformer, fit logistic regression, and evaluate against a dummy baseline.

## Generate the practice data

Run the prepared cell once. The generation code supplies the exercise data. Focus first on the resulting table. Missing values have been introduced into the four input columns.

In [25]:
import numpy as np
import pandas as pd

# Synthetic practice data: measurements at signup review, outcome over the next 90 days.
rng = np.random.default_rng(42)
n_customers = 500

df = pd.DataFrame({
    "tenure_months": rng.integers(1, 61, n_customers).astype(float),
    "monthly_charge": rng.uniform(20, 120, n_customers).round(2),
    "contract": rng.choice(["Monthly", "Annual"], n_customers),
    "signup_channel": rng.choice(["Web", "Store", "Phone"], n_customers)
})

log_odds = (
    -1.4
    - 0.04 * df["tenure_months"]
    + 0.015 * df["monthly_charge"]
    + 1.0 * (df["contract"] == "Monthly")
)
churn_probability = 1 / (1 + np.exp(-log_odds))
df["churn"] = (rng.random(n_customers) < churn_probability).astype(int)

for column in ["tenure_months", "monthly_charge", "contract", "signup_channel"]:
    missing_rows = rng.choice(n_customers, size=50, replace=False)
    df.loc[missing_rows, column] = np.nan

df.head()


,tenure_months,monthly_charge,contract,signup_channel,churn
0,6.0,85.19,Annual,Phone,1
1,47.0,106.75,Monthly,Web,1
2,40.0,65.39,Monthly,Store,1
3,27.0,44.78,Monthly,Store,0
4,26.0,43.67,NaN,NaN,1


## Inspect the schema and missing values

Type and run the following in the empty cell.

```python
df.info()
print(df.isna().sum())
print(df["churn"].value_counts())
```

Identify the numeric inputs, categorical inputs, and target. Share those groups in chat before proceeding.

In [26]:
df.info()
print(df.isna().sum())
print(df["churn"].value_counts())
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure_months   450 non-null    float64
 1   monthly_charge  450 non-null    float64
 2   contract        450 non-null    object 
 3   signup_channel  450 non-null    object 
 4   churn           500 non-null    int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 19.7+ KB
tenure_months     50
monthly_charge    50
contract          50
signup_channel    50
churn              0
dtype: int64
churn
0    363
1    137
Name: count, dtype: int64


tenure_months     float64
monthly_charge    float64
contract           object
signup_channel     object
churn               int64
dtype: object

In [27]:
numeric_features = ["tenure_months", "monthly_charge"]
categorical_features = ["contract", "signup_channel"]

## Split before learning preprocessing parameters

We will define X and y, split the rows, and fit preprocessing on training data.

The numeric pipeline will use median imputation and standard scaling. The categorical pipeline will use most-frequent imputation and one-hot encoding with unknown-category handling. ColumnTransformer will apply each pipeline to its assigned columns. We will build these steps together.

In [28]:
from sklearn.model_selection import train_test_split 

X = df[numeric_features + categorical_features]
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, random_state=42, stratify=y
)

In [30]:
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

model.fit(X_train, y_train)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
model_predictions = model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
model_accuracy = accuracy_score(y_test, model_predictions)

print("Baseline accuracy:", baseline_accuracy)
print("Model accuracy:", model_accuracy)





Baseline accuracy: 0.73
Model accuracy: 0.76


In [33]:
y_train.unique()

array([1, 0])

In [34]:
model.classes_

array([0, 1])

In [ ]:
from sklearn import confusion_matrix, classification_report

print(confusion_matrix(y_test, model_predictions, labels=[0,1]))
print(classification_report(y_test, model_predictions, zero_division=0))

[[69  4]
 [20  7]]
              precision    recall  f1-score   support

           0       0.78      0.95      0.85        73
           1       0.64      0.26      0.37        27

    accuracy                           0.76       100
   macro avg       0.71      0.60      0.61       100
weighted avg       0.74      0.76      0.72       100



for a retention campaign, the low recall means many customers who leave would receive no intervention.

logistic regression normally predicts churn when its estiated churn probability exceeds 0.5. Lowering that threshold flags more customers

In [36]:
from sklearn.model_selection import cross_val_predict

train_probabilities = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

train_probabilities

array([0.27327347, 0.14689245, 0.20259544, 0.17769187, 0.28687163,
       0.18593551, 0.20647991, 0.40024604, 0.48160371, 0.3520543 ,
       0.2844791 , 0.31139466, 0.16898632, 0.27544656, 0.08243083,
       0.31214673, 0.24362384, 0.31058131, 0.27388261, 0.55708495,
       0.17919356, 0.30234386, 0.14934688, 0.26547823, 0.19300771,
       0.3270097 , 0.4134694 , 0.11213377, 0.25377853, 0.4246042 ,
       0.53794711, 0.26476908, 0.09720461, 0.06416811, 0.23407501,
       0.2471749 , 0.22702648, 0.31117051, 0.14522076, 0.59515288,
       0.12832635, 0.11395736, 0.06449852, 0.49669345, 0.37791692,
       0.27785038, 0.42371972, 0.11814675, 0.12317693, 0.32176822,
       0.19965563, 0.23134539, 0.16547394, 0.61042817, 0.18692831,
       0.45276966, 0.4292661 , 0.332011  , 0.32663204, 0.31363562,
       0.41381705, 0.24982732, 0.13282116, 0.29264899, 0.27271504,
       0.37287659, 0.2423419 , 0.14152959, 0.12010819, 0.09984008,
       0.1539863 , 0.18163066, 0.20172945, 0.5462224 , 0.24543

In [37]:
for threshold in [0.5, 0.3]:
    predictions = (train_probabilities >= threshold).astype(int)

    print("Threshold:", threshold)
    print(classification_report(
        y_train, predictions,
        zero_division=0
    ))

Threshold: 0.5
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       290
           1       0.65      0.24      0.35       110

    accuracy                           0.76       400
   macro avg       0.71      0.59      0.60       400
weighted avg       0.73      0.76      0.71       400

Threshold: 0.3
              precision    recall  f1-score   support

           0       0.82      0.69      0.75       290
           1       0.42      0.59      0.49       110

    accuracy                           0.66       400
   macro avg       0.62      0.64      0.62       400
weighted avg       0.71      0.66      0.67       400



lowering the threshold improves churn recall at the cost of precision. I'd choose a threshold using validation results, contact costs, retention benefits and campaign capacity. 

Independent rebuild

In [41]:
df_practice = df.copy()
df_practice

features = ['tenure_months', 'monthly_charge', 'contract', 'signup_channel']

X_practice = df_practice[features]
y_practice = df_practice["churn"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_practice, 
    y_practice,
    test_size= 0.2,
    random_state=42,
    shuffle=True,
    stratify=y_practice
)

In [50]:
df_practice.dtypes

tenure_months     float64
monthly_charge    float64
contract           object
signup_channel     object
churn               int64
dtype: object

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_features_practice = ["tenure_months", "monthly_charge"]
categorical_features_practice = ["contract", "signup_channel"]

numeric_pipeline_practice = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_practice = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_practice = ColumnTransformer([
    ("numeric", numeric_pipeline_practice, numeric_features_practice),
    ("categorical", categorical_pipeline_practice, categorical_features_practice)
])

model_practice = Pipeline([
    ("preprocessor", preprocessor_practice),
    ("model", LogisticRegression(max_iter=1000))
])

model_practice.fit(X_train, y_train)
predictions_practice = model_practice.predict(X_test)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

report = classification_report(y_test, predictions_practice, zero_division=0)
accuracy = accuracy_score(y_test, predictions_practice)
consusion_matrix = confusion_matrix(y_test, predictions_practice, labels=[0, 1])
print(accuracy) 
print(report)
print(consusion_matrix)


0.76
              precision    recall  f1-score   support

           0       0.78      0.95      0.85        73
           1       0.64      0.26      0.37        27

    accuracy                           0.76       100
   macro avg       0.71      0.60      0.61       100
weighted avg       0.74      0.76      0.72       100

[[69  4]
 [20  7]]


In [57]:
model_practice.classes_

array([0, 1])